# Latency measurements


In [44]:
""" Get QoS """
import common_utils
import os
import pandas as pd
import difflib

from run_metadata import RunMetadata

# Example usage
metadata = RunMetadata()
root_folder = metadata.root_folder
filename = 'worker1.feather'
subfolders = common_utils.find_subfolders_with_file(root_folder, filename)
print(subfolders)
prom_data_paths = {os.path.basename(x): x for x in subfolders}
yolo_data_paths = {key: os.path.join(val, "worker_qos.feather") for key, val in prom_data_paths.items()}


['../../../data_warehouse/minimized_warehouse_7cc\\1738881900_(1.1000)', '../../../data_warehouse/minimized_warehouse_7cc\\1738915261_(1.5000)', '../../../data_warehouse/minimized_warehouse_7cc\\1738948257_(1.10000)']


In [45]:
from utils.header_cleaner import *
import difflib
import os


"""
Fetch paths to the data
"""


"""
Get corresponding yolo stats for each model 
"""
response_time = {}
for key in prom_data_paths.keys():
    try:
        yolo_df = common_utils.read_feather_cached(yolo_data_paths[key])
    except:
        print(f"Failed to read {key}")
        continue
    yolo_df['total_inference_time'] = yolo_df['inf'] + yolo_df['post'] + yolo_df['pre']
    yolo_df['end_to_end_response_time'] = yolo_df['total_inference_time'] + yolo_df['queue']
    yolo_df['start'] = pd.to_datetime(yolo_df['start_time'], unit='ms')  # Convert to datetime (optional)
    yolo_df.set_index('start', inplace=True)
    resampled_df = yolo_df.resample('5s')
    model_info = common_utils.path_to_workers_and_pcl_size(key)
    if model_info.resolution not in response_time:
        response_time[model_info.resolution] = {}
    response_time[model_info.resolution][model_info.num_vehicles] = resampled_df.agg({'end_to_end_response_time': 'mean'}).reset_index()['end_to_end_response_time'].rename(key)


In [50]:
import plotly.express as px
import pandas as pd
import numpy as np

# List to store the aggregated metrics
aggregated_metrics = []

# Adding Debug Prints
print("Starting calculation of mean latency and jitter for each resolution...")
print(f"Initial response_time keys (resolutions): {list(response_time.keys())}")

# Calculate mean latency and jitter for each resolution
for resolution, models in response_time.items():
    print(f"\nProcessing resolution: {resolution}")

    try:
        # Combine all models for the current resolution
        combined_df = pd.DataFrame.from_dict(models)

        # Debug: Print shape and first few rows of combined DataFrame
        print(f"Shape of combined DataFrame for resolution {resolution}: {combined_df.shape}")
        print(f"First few rows of combined DataFrame for resolution {resolution}:\n{combined_df.head()}")

        # Ensure there are valid (non-NaN, non-zero) values for calculations
        if combined_df.empty or combined_df.isna().all().all():
            print(f"Combined DataFrame for resolution {resolution} is empty or only contains NaN values. Skipping...")
            continue

        # Handle NaN values by dropping them row-wise, if needed
        combined_df = combined_df.dropna(how='all')  # Drop rows with all NaN
        if combined_df.empty:
            print(f"All rows dropped due to NaN values for resolution {resolution}. Skipping...")
            continue

        # Compute mean latency
        mean_across_time_points = combined_df.mean(axis=1)  # Mean across time points
        mean_latency = mean_across_time_points.mean()  # Mean across workers

        # Compute jitter (robust calculation):
        # Step 1: Aggregate within each model (standard deviation per column)
        jitter_within_models = combined_df.std(axis=0)  # Standard deviation per model
        # Step 2: Aggregate across models (mean of standard deviations)
        jitter = jitter_within_models.mean()

        # Store metrics for this resolution
        aggregated_metrics.append({"Resolution": resolution, "Metric Type": "Latency", "Value": mean_latency})
        aggregated_metrics.append({"Resolution": resolution, "Metric Type": "Jitter", "Value": jitter})

        # Debug outputs for calculated metrics
        print(f"Mean Latency for resolution {resolution}: {mean_latency}")
        print(f"Jitter for resolution {resolution}: {jitter}")

    except Exception as e:
        print(f"Error while processing resolution {resolution}: {e}")
        continue

# Debug: Print aggregated metrics
print("\nAggregated Metrics:")
for metric in aggregated_metrics:
    print(metric)

# Create a DataFrame from the aggregated metrics
metrics_df = pd.DataFrame(aggregated_metrics)

# Debug: Print final DataFrame
print("\nFinal metrics DataFrame:")
print(metrics_df)

# Create a grouped bar plot with Plotly Express
fig = px.bar(
    metrics_df,
    x="Resolution",
    y="Value",
    color="Metric Type",
    barmode="group",
    title="Mean Latency and Jitter Metrics by Resolution",
    log_y=True,
    error_y=metrics_df["Value"].apply(lambda x: x * 0.1 if not np.isnan(x) else 0)  # Adding 10% as error bars
)

# Customize axes labels
fig.update_layout(
    xaxis_title="Resolution",
    yaxis_title="Value (log scale)"
)

# Show the plot
fig.show()

Starting calculation of mean latency and jitter for each resolution...
Initial response_time keys (resolutions): [1000, 5000, 10000]

Processing resolution: 1000
Shape of combined DataFrame for resolution 1000: (5910, 1)
First few rows of combined DataFrame for resolution 1000:
             1
0  2199.268893
1          NaN
2          NaN
3          NaN
4          NaN
Mean Latency for resolution 1000: 2060.700200542584
Jitter for resolution 1000: 7409.418158346073

Processing resolution: 5000
Shape of combined DataFrame for resolution 5000: (5912, 1)
First few rows of combined DataFrame for resolution 5000:
             1
0  2594.196567
1          NaN
2          NaN
3          NaN
4          NaN
Mean Latency for resolution 5000: 2701.828979298031
Jitter for resolution 5000: 7829.107791522974

Processing resolution: 10000
Shape of combined DataFrame for resolution 10000: (5913, 1)
First few rows of combined DataFrame for resolution 10000:
             1
0  2864.840768
1          NaN
2    

In [47]:
import plotly.express as px
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# List to store the aggregated metrics
aggregated_metrics = []

# For each resolution, compute mean latency and jitter (standard deviation)
for resolution, models in response_time.items():
    # Combine all models for the current resolution
    combined_df = pd.DataFrame.from_dict(models)

    # Compute mean latency and mean jitter across all time points and workers
    mean_latency = combined_df.mean(axis=1).mean()  # mean across time points, then mean across workers
    jitter = combined_df.std(axis=1).mean()        # standard deviation across time points, then mean across workers

    # Store results
    aggregated_metrics.append({'Resolution': resolution, 'Mean Latency': mean_latency, 'Jitter': jitter})

# Create a DataFrame from the collected metrics
metrics_df = pd.DataFrame(aggregated_metrics)

# Create subplots for latency and jitter
fig = make_subplots(
    rows=1, cols=2, subplot_titles=("Mean Latency", "Jitter"),
    shared_yaxes=False  # Separate axes for latency and jitter
)

# Add bar plot for Mean Latency
fig.add_trace(
    go.Bar(x=metrics_df['Resolution'], y=metrics_df['Mean Latency'], name='Mean Latency'),
    row=1, col=1
)

# Add bar plot for Jitter
fig.add_trace(
    go.Bar(x=metrics_df['Resolution'], y=metrics_df['Jitter'], name='Jitter'),
    row=1, col=2
)

# Update layout and axis labels
fig.update_yaxes(title_text='Mean Latency (ms)', type="log", row=1, col=1)
fig.update_yaxes(title_text='Jitter (ms)', type="log", row=1, col=2)
fig.update_xaxes(title_text='Resolution', row=1, col=1)
fig.update_xaxes(title_text='Resolution', row=1, col=2)
fig.update_layout(
    title_text='Aggregated Latency and Jitter Metrics',
    height=600,
    showlegend=False
)

# Show the plot
fig.show()

In [48]:
"""
import os
import pandas as pd
from functools import lru_cache
from collections import namedtuple
import difflib
import plotly.express as px

response_time = {}
for key in prom_data_paths.keys():
    try:
        yolo_df = common_utils.read_feather_cached(yolo_data_paths[key])
    except:
        print(f"Failed to read {key}")
        continue
    yolo_df['total_inference_time'] = yolo_df['inf'] + yolo_df['post'] + yolo_df['pre']
    yolo_df['end_to_end_response_time'] = yolo_df['total_inference_time'] + yolo_df['queue']
    yolo_df['start'] = pd.to_datetime(yolo_df['start_time'], unit='ms')
    yolo_df.set_index('start', inplace=True)
    resampled_df = yolo_df.resample('5s')
    model_info = common_utils.path_to_workers_and_pcl_size(key)
    if model_info.resolution not in response_time:
        response_time[model_info.resolution] = {}
    response_time[model_info.resolution][model_info.num_vehicles] = resampled_df.agg({'end_to_end_response_time': 'min'}).reset_index()['end_to_end_response_time'].rename(key)

data = []
for resolution, models in response_time.items():
    for model, latency in models.items():
        if not latency.empty:
            min_latency = latency.min()
            data.append((str(resolution), model, min_latency))

# Sort data before creating the DataFrame
# sorted_data = sorted(data, key=lambda x: common_utils._sort_key_size_version(x[1]))

df = pd.DataFrame(data, columns=['Resolution', 'Model', 'Min Latency'])
fig = px.bar(df, x='Model', y='Min Latency', color='Resolution', barmode="group", title=f'Minimum end-to-end latency', log_y=True)
fig.update_layout(yaxis_title='Min Latency (ms)')
fig.show()"""

'\nimport os\nimport pandas as pd\nfrom functools import lru_cache\nfrom collections import namedtuple\nimport difflib\nimport plotly.express as px\n\nresponse_time = {}\nfor key in prom_data_paths.keys():\n    try:\n        yolo_df = common_utils.read_feather_cached(yolo_data_paths[key])\n    except:\n        print(f"Failed to read {key}")\n        continue\n    yolo_df[\'total_inference_time\'] = yolo_df[\'inf\'] + yolo_df[\'post\'] + yolo_df[\'pre\']\n    yolo_df[\'end_to_end_response_time\'] = yolo_df[\'total_inference_time\'] + yolo_df[\'queue\']\n    yolo_df[\'start\'] = pd.to_datetime(yolo_df[\'start_time\'], unit=\'ms\')\n    yolo_df.set_index(\'start\', inplace=True)\n    resampled_df = yolo_df.resample(\'5s\')\n    model_info = common_utils.path_to_workers_and_pcl_size(key)\n    if model_info.resolution not in response_time:\n        response_time[model_info.resolution] = {}\n    response_time[model_info.resolution][model_info.num_vehicles] = resampled_df.agg({\'end_to_end_re